In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import keras
import keras.ops as K
from keras.layers import Input, Flatten, Dense
from keras.optimizers import Adam
from keras.metrics import BinaryAccuracy

# from keras.models import Sequential
from deel.lip.model import Sequential

from deel.lip.layers import (
    SpectralDense,
    SpectralConv2D,
    ScaledL2NormPooling2D,
    FrobeniusDense,
)
from deel.lip.activations import GroupSort, GroupSort2
from deel.lip.losses import HKR, KR, HingeMargin, MulticlassHKR, MulticlassKR

import numpy as np
import pandas as pd

from lipschitz_optimization_tools import get_local_maximum_multiclass, echantillonner_boule_l2_simple, create_difference_model

import sys
sys.path.append('..')

from radius_evaluation_tools import single_compute_relaxation_radius_multiclass


from data_processing import load_data, select_data_for_radius_evaluation
from radius_evaluation_tools import compute_binary_certificate, starting_point_dichotomy
from data_processing_torch import *
from notebooks_creation_models.VGG_Arthur import *
import torch
import yaml
import pickle

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
sys.path.append('/home/aws_install/robustess_project')
import liresnet.models as models

In [4]:
with open('./../notebooks_creation_models/config.yaml', 'r') as f:
    # maybe dupplicate config
    cfg = yaml.safe_load(f)
_, test_loader = load_cifar10(cfg)

/home/aws_install/miniconda3/envs/k3torchenv/lib/python3.10/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [5]:
print("loading model :")
weights = torch.load('/home/aws_install/robustess_project/lip_models/cifar10-12x512_799.pth').get('backbone')
with open('/home/aws_install/robustess_project/liresnet/configs/cifar10.yaml', 'r') as f:
        cfg = yaml.load(f, Loader=yaml.Loader)
model_cfg = cfg['model']
dataset_cfg = cfg['dataset']
gloro_cfg = cfg['gloro']
model = models.GloroNet(**model_cfg, **dataset_cfg).to(device)
model.load_state_dict(weights)
model.eval()


loading model :


GloroNet(
  (stem): Sequential(
    (0): Conv2d(3, 512, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2), output_padding=(1, 1))
    (1): MinMax(dim=1)
  )
  (conv): LiResConv(
    depth=12, width=512, centering=True
    (act): MinMax(dim=1)
  )
  (neck): Map2Vec(
    (activation): MinMax(dim=1)
  )
  (linear): LiResMLP(
    depth=8, width=2048
    (act): MinMax(dim=1)
  )
  (head): head(in_features=2048, out_features=10, bias=True)
)

In [6]:
from keras.layers import TorchModuleWrapper, Input

In [7]:
layer_torch = TorchModuleWrapper(model)

In [8]:
k_model = keras.models.Sequential([Input((3,32,32)), layer_torch])

In [9]:
class LiResNet(keras.Model):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.device = next(model.parameters()).device
        self.output_shape = (None, 10)
    def build(self, input_shape):
        """Dynamically infers the output shape."""
        # c, h, w = input_shape[1], input_shape[2], input_shape[3]
        # dummy_input = torch.randn(1, c, h, w, device=self.device)
        # with torch.no_grad():
        #     dummy_output = self.model(dummy_input)
        # _, out = dummy_output.shape
        # # Define the Keras-compatible output shape attribute
        # self.output_shape_keras = (None, out_c, out_h, out_w)
        super().build(input_shape)
    def call(self, x):
        return self.model(x)

In [10]:
keras_model = LiResNet(model=model)

In [11]:
keras_model.summary()

Model: "li_res_net"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ torch_module_wrapper_1          │ ?                      │    82,944,010 │
│ (TorchModuleWrapper)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 82,944,010 (316.41 MB)

 Trainable params: 82,944,010 (316.41 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
(x_train, y_train), _ = keras.datasets.cifar10.load_data()
x_train = x_train.astype("float32") / 255
y_train = keras.utils.to_categorical(y_train)

In [13]:
keras_model(x_train[:10])

tensor([[10.8687,  8.0230, 17.5902, 20.1534, 18.8996, 20.2226, 27.4561, 14.9714,
          9.4883,  7.5412],
        [12.6718, 18.2049,  9.4101,  9.7515,  5.4579, 10.9741,  5.6167,  8.6645,
         14.5670, 37.6884],
        [19.2139, 13.5586, 12.2912, 11.3462, 13.5775,  7.2453,  7.5559, 14.4153,
         19.1316, 31.2346],
        [11.5448,  7.3122, 16.0740, 14.3902, 26.4946, 14.9300, 21.6906, 13.1727,
          7.9756,  7.4416],
        [13.6287, 37.9883,  2.2351,  2.1648,  7.5931, -0.2325,  3.5222,  3.0971,
          7.4788, 16.8582],
        [ 6.6145, 30.3660,  7.4350, 10.7789,  7.3294, 10.9712,  9.5803,  8.6843,
          7.2781, 15.9467],
        [15.3517,  8.7146, 30.5510, 15.9660, 20.0403, 14.7800, 17.8254, 17.0212,
          7.6641, 10.4726],
        [ 7.2085,  1.6848, 11.9177, 13.0749, 27.2666, 18.3878,  8.2678, 45.1891,
          0.4338,  5.8170],
        [23.4189, 11.8446, 16.2451, 11.0775, 11.7976,  8.0213, 10.6617,  6.3507,
         38.9523, 10.6660],
        [14.8963, 1

In [14]:
from lipschitz_optimization_tools import get_local_maximum_multiclass

In [15]:
x_sample = x_train[6:7].flatten()

In [16]:
label = (y_train[6]).argmax()

In [29]:
eps = 0.2

In [30]:
y_list = []
for _ in range(100):
        y_list.append(echantillonner_boule_l2_simple(x_sample, eps))

In [31]:
x_sample.shape

(3072,)

In [32]:
get_local_maximum_multiclass(x_sample, label, eps, y_list, k_model, input_shape = (3,32,32))

('optimal',
 -1.2104132684909068,
 array([0.63810118, 0.40551133, 0.46185351, ..., 0.11769926, 0.10568386,
        0.1732774 ]))

In [42]:
output_dir = "./../benchmark_dataset"
images_path = os.path.join(output_dir, "images.pkl")
targets_path = os.path.join(output_dir, "targets.pkl")

# --- Load the Tensors ---
print(f"Loading data from {output_dir}...")

# Load the images tensor
with open(images_path, 'rb') as f:
        images = pickle.load(f)

# Load the targets tensor
with open(targets_path, 'rb') as f:
    labels = pickle.load(f)
        
images = images.cpu().detach().numpy()
labels = labels.cpu().detach().numpy()

Loading data from ./../benchmark_dataset...


In [43]:
x_sample = images[6:7].flatten()

In [44]:
label = (labels[6]).argmax()

In [45]:
get_local_maximum_multiclass(x_sample, label, eps, y_list, k_model, input_shape = (3,32,32))

/home/aws_install/miniconda3/envs/k3torchenv/lib/python3.10/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor']
Received: inputs=Tensor(shape=(1, 3, 32, 32))
  warnings.warn(msg)
/home/aws_install/miniconda3/envs/k3torchenv/lib/python3.10/site-packages/cvxpy/problems/problem.py:1510: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(


('optimal_inaccurate',
 -16.69989557459696,
 array([0.44853443, 0.44750961, 0.44685327, ..., 0.55058367, 0.55470553,
        0.56175155]))